In [ ]:
import pandas as pd
import numpy as np
# load data
df = pd.read_csv('../data/cleaned/wednesday_cleaned.csv')
print(f'Shape: {df.shape}')

# get columns that are numberic 
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Numeric columns: {len(numeric_cols)}')

Shape: (61001, 69)
Numeric columns: 68


In [5]:
# compute mean, std, min, max for all features
stats = pd.DataFrame({
    'mean': df[numeric_cols].mean(),
    'std': df[numeric_cols].std(),
    'min': df[numeric_cols].min(),
    'max': df[numeric_cols].max()
})

print('Stats:')
print(stats)

Stats:
                                     mean           std  min          max
Destination_Port             7.592881e+02  5.734532e+03  0.0      63276.0
Flow_Duration                4.189280e+07  4.263155e+07  1.0  119991902.0
Total_Fwd_Packets            5.866887e+00  7.099562e+01  1.0      17487.0
Total_Backward_Packets       4.246668e+00  9.556690e+01  0.0      23539.0
Total_Length_of_Fwd_Packets  3.848013e+02  1.811250e+03  0.0     278248.0
...                                   ...           ...  ...          ...
Idle_Mean                    3.677495e+07  4.274586e+07  0.0  120000000.0
Idle_Std                     9.960781e+05  5.291552e+06  0.0   60800000.0
Idle_Max                     3.775972e+07  4.280229e+07  0.0  120000000.0
Idle_Min                     3.596751e+07  4.304179e+07  0.0  120000000.0
Attack                       9.180833e-01  2.742400e-01  0.0          1.0

[68 rows x 4 columns]


In [3]:
# identify low and zero variance features
variance = df[numeric_cols].var()

zero_var = []
low_var = []

for col in numeric_cols:
    var_val = variance[col]
    if var_val == 0:
        zero_var.append(col)
    elif var_val < 0.01:
        low_var.append(col)

print(f'Zero variance features: {len(zero_var)}')
print(zero_var)

print(f'\nLow variance features (var < 0.01): {len(low_var)}')
for col in low_var:
    print(f'  {col}: variance = {variance[col]:.6f}')

Zero variance features: 0
[]

Low variance features (var < 0.01): 2
  RST_Flag_Count: variance = 0.000033
  ECE_Flag_Count: variance = 0.000033


In [6]:
# check which features are skewed
skewness = df[numeric_cols].skew()

# find features with high skewness absolute val over 1
skewed_features = []
skew_values = []

for col in numeric_cols:
    skew_val = skewness[col]
    if skew_val > 1 or skew_val < -1:
        skewed_features.append(col)
        skew_values.append(skew_val)

# sort by absolute value of skewness with highest first
sorted_pairs = sorted(zip(skewed_features, skew_values), key=lambda x: abs(x[1]), reverse=True)
skewed_features = [pair[0] for pair in sorted_pairs]
skew_values = [pair[1] for pair in sorted_pairs]

print('Highly skewed features (skew > 1 or < -1):', len(skewed_features))
print('\nTop 10 most skewed:')
for i in range(10):
    if i < len(skewed_features):
        print(str(i+1) + '. ' + skewed_features[i] + ': ' + str(round(skew_values[i], 3)))

Highly skewed features (skew > 1 or < -1): 48

Top 10 most skewed:
1. act_data_pkt_fwd: 245.906
2. Subflow_Bwd_Bytes: 245.899
3. Total_Length_of_Bwd_Packets: 245.899
4. Total_Backward_Packets: 244.864
5. Subflow_Bwd_Packets: 244.864
6. Total_Fwd_Packets: 244.75
7. Subflow_Fwd_Packets: 244.75
8. Bwd_Header_Length: 243.212
9. Fwd_Header_Length: 242.962
10. RST_Flag_Count: 174.64


12/1/2025 - Umar - Univariate Analysis

**Purpose:**  
Compute summary statistics for all numeric features and identify possible issues with variance and distribution skewness.

**Interpretation / Findings:** 
- **Basic Statistics:** Computed mean, std, min, and max for all features
- **Zero Variance Features:** 0 features with zero variance (all features have variation)
- **Low Variance Features:** 2 features with variance < 0.01:
  - RST_Flag_Count: variance = 0.000033
  - ECE_Flag_Count: variance = 0.000033
  - Not sure if these would be useful for modeling in this context
- **Skewed Distributions:** Many features show high skewness (|skew| > 1)
  - Features related to flow bytes, packet sizes, and timing show extreme skewness
  - I feel like this may be expected in network traffic 
  - May need transformation 

**Notes for Team:**
Consider removing or combining the low variance flagged features. Skewed features may need some kind of transformation before modeling.

### Done with Task